# Klasifikasi Bagian Tanaman — CNN, 5 Kelas

**Dataset:** `dataset_tanaman_final_5kelas.zip` (2.500 gambar, masing-masing 500 untuk `akar`, `batang`, `buah`, `bunga`, `daun`).**Pembagian:** train 350, validation 75, test 75 per kelas.**Input:** RGB 224×224. **Label:** urutan alfabet otomatis dari direktori.

## Sebelum menjalankan
1. Upload file `dataset_tanaman_final_5kelas.zip` ke **MyDrive** di Google Drive.
2. Di Google Colab pilih **Runtime → Change runtime type → GPU** bila tersedia.
3. Jalankan semua cell dari atas ke bawah. Tidak perlu menyalin 2.500 JPG satu per satu.

> **Catatan kualitas:** beberapa gambar pada kelas `akar` tampak seperti potongan cabang/kayu dan kelas akar mengandung beberapa varian dari foto sumber yang sama. Periksa `review_akar_*.jpg` sebelum menginterpretasikan akurasi. Metadata foto sumber dan audit tersedia di ZIP dataset.

## 1. Hubungkan Google Drive dan import library

In [ ]:
from google.colab import drive, files
from pathlib import Path
import csv
import json
import zipfile
import shutil

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EXPECTED_CLASSES = ["akar", "batang", "buah", "bunga", "daun"]
EXPECTED_PER_CLASS = {"train": 350, "val": 75, "test": 75}

tf.keras.utils.set_random_seed(SEED)
drive.mount('/content/drive')
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 2. Ekstrak ZIP dataset langsung ke Colab
Simpan file ZIP pada `MyDrive/dataset_tanaman_final_5kelas.zip`. Dataset diekstrak ke disk Colab agar pembacaan gambar lebih cepat dibanding membaca file satu per satu dari Google Drive.

In [ ]:
ZIP_PATH = Path('/content/drive/MyDrive/dataset_tanaman_final_5kelas.zip')
FOLDER_DRIVE = Path('/content/drive/MyDrive/dataset_tanaman_final')
BASE_LOCAL = Path('/content/dataset_tanaman_final')

if ZIP_PATH.is_file():
    if BASE_LOCAL.exists():
        shutil.rmtree(BASE_LOCAL)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall('/content')
    BASE = BASE_LOCAL
    print('Dataset diekstrak dari:', ZIP_PATH)
elif FOLDER_DRIVE.is_dir():
    BASE = FOLDER_DRIVE
    print('Menggunakan folder dari Google Drive:', BASE)
else:
    raise FileNotFoundError(
        'Upload dataset_tanaman_final_5kelas.zip ke MyDrive, atau buat folder MyDrive/dataset_tanaman_final'
    )

for split, expected in EXPECTED_PER_CLASS.items():
    print('\nSplit:', split)
    for cls in EXPECTED_CLASSES:
        folder = BASE / split / cls
        paths = list(folder.glob('*.jpg'))
        print(f'  {cls:8s}: {len(paths):3d}')
        assert len(paths) == expected, f'{split}/{cls}: diharapkan {expected}, ditemukan {len(paths)}'

print('\nDataset lengkap: 5 kelas, 2.500 gambar.')

## 3. Load dataset sebagai TensorFlow Dataset
Label kelas dibaca otomatis dari urutan folder alfabet. **Normalisasi 0–1 ditempatkan di dalam model**, bukan di loader maupun di FastAPI, agar konsisten ketika melakukan prediksi.

In [ ]:
def get_ds(split: str, shuffle: bool):
    return tf.keras.utils.image_dataset_from_directory(
        BASE / split,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=SEED if shuffle else None,
        label_mode='int',
        color_mode='rgb',
        interpolation='bilinear',
    )

train_ds_raw = get_ds('train', True)
val_ds_raw = get_ds('val', False)
test_ds_raw = get_ds('test', False)

class_names = list(train_ds_raw.class_names)
assert class_names == EXPECTED_CLASSES, f'Urutan kelas tidak cocok: {class_names}'
assert list(val_ds_raw.class_names) == class_names
assert list(test_ds_raw.class_names) == class_names

AUTO = tf.data.AUTOTUNE
train_ds = train_ds_raw.prefetch(AUTO)
val_ds = val_ds_raw.prefetch(AUTO)
test_ds = test_ds_raw.prefetch(AUTO)

print('Urutan kelas:', class_names)
print('Jumlah kelas:', len(class_names))

## 4. Lihat contoh data sebelum training

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
images, labels = next(iter(train_ds_raw))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].numpy().astype('uint8'))
    ax.set_title(class_names[int(labels[i])])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Model CNN (baseline)
Lapisan augmentasi hanya aktif saat training. Lapisan `Rescaling(1./255)` ada **di dalam model**, sehingga FastAPI harus mengirim nilai piksel RGB `0–255` tanpa membaginya lagi.

In [ ]:
from tensorflow.keras import layers

model = tf.keras.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.Rescaling(1.0 / 255.0),

    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.30),
    layers.Dense(len(class_names), activation='softmax'),
], name='cnn_bagian_tanaman_5_kelas')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

## 6. Training dan validasi
Data **test** tidak dipakai untuk memilih model atau hyperparameter. `EarlyStopping` mengembalikan bobot terbaik berdasarkan validation loss.

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=callbacks,
)

## 7. Grafik hasil training

In [ ]:
RESULTS = Path('/content/results_cnn_5kelas')
RESULTS.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history['accuracy'], label='Train')
ax[0].plot(history.history['val_accuracy'], label='Validation')
ax[0].set_title('Accuracy')
ax[0].set_xlabel('Epoch')
ax[0].legend()

ax[1].plot(history.history['loss'], label='Train')
ax[1].plot(history.history['val_loss'], label='Validation')
ax[1].set_title('Loss')
ax[1].set_xlabel('Epoch')
ax[1].legend()
plt.tight_layout()
plt.savefig(RESULTS / 'training_curves.png', dpi=160, bbox_inches='tight')
plt.show()

## 8. Evaluasi satu kali pada test set
Hasil evaluasi test dapat dipakai pada paper UAS: accuracy, precision, recall, F1-score, dan confusion matrix.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)
y_prob = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

report_text = classification_report(
    y_true, y_pred, target_names=class_names, digits=4, zero_division=0
)
report_dict = classification_report(
    y_true, y_pred, target_names=class_names,
    digits=4, zero_division=0, output_dict=True
)
print('Test accuracy:', round(float(test_accuracy), 4))
print(report_text)
(RESULTS / 'classification_report.txt').write_text(report_text, encoding='utf-8')
(RESULTS / 'metrics.json').write_text(
    json.dumps({'test_loss': float(test_loss), 'test_accuracy': float(test_accuracy),
                'classification_report': report_dict}, indent=2, ensure_ascii=False),
    encoding='utf-8'
)

cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(cmap='Blues', ax=ax, colorbar=False)
plt.title('Confusion Matrix - 5 Kelas')
plt.tight_layout()
plt.savefig(RESULTS / 'confusion_matrix.png', dpi=160, bbox_inches='tight')
plt.show()

## 9. Simpan model untuk FastAPI
Keluarkan **dua file utama**: `model_tanaman.keras` dan `class_names.json`. Backend dari starter project sebelumnya sudah membaca urutan kelas secara dinamis; jumlah output model harus 5.

In [ ]:
EXPORT = Path('/content/export_model_tanaman_5kelas')
EXPORT.mkdir(parents=True, exist_ok=True)
model.save(EXPORT / 'model_tanaman.keras')
(EXPORT / 'class_names.json').write_text(
    json.dumps(class_names, ensure_ascii=False, indent=2), encoding='utf-8'
)

# Uji file yang diekspor, sebelum digunakan oleh FastAPI.
loaded = tf.keras.models.load_model(EXPORT / 'model_tanaman.keras', compile=False)
assert int(loaded.output_shape[-1]) == len(class_names) == 5
print('Model diekspor, urutan label:', class_names)

export_zip = Path('/content/model_tanaman_export_5kelas.zip')
with zipfile.ZipFile(export_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(EXPORT / 'model_tanaman.keras', arcname='model_tanaman.keras')
    z.write(EXPORT / 'class_names.json', arcname='class_names.json')
    for path in RESULTS.iterdir():
        if path.is_file():
            z.write(path, arcname=f'results/{path.name}')
print('File untuk diunduh:', export_zip)
files.download(str(export_zip))

## 10. Integrasi dengan FastAPI + Next.js

Ekstrak `model_tanaman_export_5kelas.zip`, kemudian salin `model_tanaman.keras` dan `class_names.json` ke folder **`backend/models/`** dari proyek Anda.

```text
backend/models/
  model_tanaman.keras
  class_names.json
```

Jalankan backend dari folder `backend`:

```powershell
python -m uvicorn app.main:app --reload
```

Buka `http://127.0.0.1:8000/health` untuk memeriksa model, `http://127.0.0.1:8000/docs` untuk mencoba `POST /predict`, lalu jalankan frontend (`npm run dev`) dan buka `http://localhost:3000`.

**Penting:** jika JSON health masih `model_missing`, artinya kedua file model belum berada pada folder yang benar. Model klasifikasi hanya mengembalikan satu bagian tanaman utama, bukan menandai beberapa organ sekaligus.